## RAG 파이프라인 구축

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

load_dotenv()

True

In [2]:
model = init_chat_model("openai:gpt-5.6-luna")

# 임베딩 및 저장
DB_PATH = "../data/k_ladder_2026"

# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 저장된 벡터 DB 가져와야
load_vs = Chroma(
    collection_name="k_ladder_2026",
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

In [ ]:
# 검색기
retriever_mmr = load_vs.as_retriever(search_type="mmr", 
                                 search_kwargs={"k": 5, "fetch_k": 50, "lamda_mult" : 0.25})



In [5]:
# RAG로 붙여보기

SYSTEM_PROMPT = """
너는 공공정책 안내 도우미다.
아래 자료를 참고해서 답해라.
참고 자료에 없으면 "자료에 없음" 이라고 말해라.
정확한 자격, 금액, 기한은 공고 확인이 필요하다고 꼭 덧붙여라
답 끝에 참고한 페이지 번호를 [p.60]과 같은 형식으로 표시해라.
"""

# 검색 결과 문서를 받았을 때 메타데이터와 내용을 합쳐서 텍스트로 반환하는 함수 작성
def format_docs(docs):
    context = ""

    for doc in docs:
        context += f"[p.{doc.metadata["page"]}] \n {doc.page_content} \n\n"

    return context

In [ ]:
# 검색기 넣은 뒤 검색 결과 문서를 받았을 때 메타데이터와 내용을 합쳐서 텍스트로 반환하는 함수에 넣는 과정을 정리
chain = retriever_mmr | format_docs # 검색한 결과를 context로 변환
chain

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000023AE4450770>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 50, 'lamda_mult': 0.25})
| RunnableLambda(format_docs)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# RAG 프롬프트 완성해보기
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ('human', "참고자료\n{context} \n 질문 {question}")
])

# 순서: 먼저 문장이 리트리버 들어가고 formatdocs 통과해 context가 되고, 그와 별개로 문장이 question에 바로 들어가 형성된 뒤 rag_prompt에 또 들어간다!

rag_chain = (# question은 검색기를 거쳐 context라는 key의 value가 돼야 하고, 
    {"context" : (retriever_mmr | format_docs), "question" : RunnablePassthrough()} 
    | rag_prompt    # invoke안의 내용 그대로 prompt에 들어가야
  )  

rag_chain.invoke("청년 월세 지원 정책 찾아줘")

ChatPromptValue(messages=[SystemMessage(content='\n너는 공공정책 안내 도우미다.\n아래 자료를 참고해서 답해라.\n참고 자료에 없으면 "자료에 없음" 이라고 말해라.\n정확한 자격, 금액, 기한은 공고 확인이 필요하다고 꼭 덧붙여라\n답 끝에 참고한 페이지 번호를 [p.60]과 같은 형식으로 표시해라.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='참고자료\n[p.39] \n 037모두의 정책 K-희망사다리 2026\n여성청소년 \n생리용품 지원\n지원대상 \t • \t기초생활수급(생계·의료·주거·교육급여),\t법정차상위계층,\t한부모가족\t지원\t\n대상\t가구의\t9~24세\t여성청소년\n핵심내용 \t •\t 여성청소년\t생리용품\t바우처\t지원(월\t1만\t4,000원),\t국민행복카드로\t구매\n •9세가\t되는\t해의\t1월\t1일부터\t24세가\t끝나는\t해의\t12월\t31일까지\t지원\t\n이용방법 \t •\t 온라인\t신청:\t복지로(www.bokjiro.go.kr)\t또는\t모바일\t앱\n\t •방문\t신청:\t읍·면·동\t주민센터\t및\t행정복지센터\t\n  ※  지원대상 결정 전·후 청소년 본인 또는 신청인(바우처 신청서상의 신청인) 명의의 국민\n행복카드를 발급받아야 사용 가능 \n문의처\t• 성평등가족부\t청소년정책과(☎02-2100-6242)\n\t •한국사회보장정보원(☎1566-3232)\n\t •읍·면·동\t주민센터\t및\t행정복지센터\t\t\n02-2100-6242\n성평등가족부 청소년정책과\n1만  4,000원\n월\t지원금 \n\n[p.14] \n 청년미래적금\n 1600-5500\n금융위원회\n012따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도\n지원대상 \t •\t 일정\t소득\t이하\t만\t19~34세\t청년(병역\t최대\t6년\t인정)\n\t \t - \t\t일반형:\t개인\t소

In [12]:
rag_chain = (# question은 검색기를 거쳐 context라는 key의 value가 돼야 하고, 
    {"context" : (retriever_mmr | format_docs), "question" : RunnablePassthrough()} 
    | rag_prompt  # invoke안의 내용 그대로 prompt에 들어가야
    | model
    | StrOutputParser()
)

rag_chain.invoke("청년 월세 지원 정책 찾아줘")

'참고자료에는 **청년 월세 지원의 구체적인 자격·지원금액·신청기간**이 나와 있지 않아 **자료에 없음**입니다.\n\n다만 **혜택알리미**를 이용하면 개인정보 활용 동의 후 소득·재산, 주거 상황 등을 분석해 청년월세 지원 등 본인에게 해당할 가능성이 높은 공공서비스를 맞춤 추천받고, 일부 서비스는 신청까지 연계할 수 있습니다. 혜택알리미는 대한민국 국민 누구나 이용 대상입니다.\n\n정확한 자격, 금액, 기한은 반드시 해당 연도 공고를 확인해야 합니다. [p.256]'

#### 구조화된 출력으로 출력 받기

In [14]:
# 구조화된 출력으로 출력 받기
from pydantic import BaseModel, Field

class AnswerStyle(BaseModel):
    answer : str = Field(description="최종 답변")
    source : str = Field(description="출처")

structured_model = model.with_structured_output(AnswerStyle, method="json_schema")

rag_chain = (
    # question은 검색기를 거쳐 context라는 key의 value가 돼야 하고, 
    {"context" : (retriever_mmr | format_docs), "question" : RunnablePassthrough()} 
    | rag_prompt  # invoke안의 내용 그대로 prompt에 들어가야
    | structured_model
    # | StrOutputParser() # 얘는 삭제
)

result = rag_chain.invoke("청년 월세 지원 정책 찾아줘")
print(result)

answer='청년 월세 지원의 구체적인 지원대상·지원금액·신청기간·신청방법은 참고자료에 없습니다. 다만 ‘혜택알리미’에서 개인정보 활용에 동의하면 소득·재산 및 거주 상황 등을 분석해 청년월세 지원 대상 여부를 맞춤 추천하고, 신청까지 연계할 수 있습니다. 혜택알리미는 대한민국 모든 국민이 이용할 수 있습니다. 정확한 자격, 금액, 기한은 관련 사업 공고를 확인해야 합니다.' source='[p.256]'


In [15]:
result.model_dump()

{'answer': '청년 월세 지원의 구체적인 지원대상·지원금액·신청기간·신청방법은 참고자료에 없습니다. 다만 ‘혜택알리미’에서 개인정보 활용에 동의하면 소득·재산 및 거주 상황 등을 분석해 청년월세 지원 대상 여부를 맞춤 추천하고, 신청까지 연계할 수 있습니다. 혜택알리미는 대한민국 모든 국민이 이용할 수 있습니다. 정확한 자격, 금액, 기한은 관련 사업 공고를 확인해야 합니다.',
 'source': '[p.256]'}

### 실습 - 아래 데이터로 RAG 구축
- 16-1_K희망사다리2026_모두의정책.pdf
- 16-5_Samsung_Electronics_Sustainability_Report_2026_KOR.pdf
- 16-6_개인정보FAQ.csv